In [1]:
!pip install transformers

In [2]:
!pip install "transformers[torch]"

In [3]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [4]:
path = "/content/drive/MyDrive/Colab Notebooks/Transformer"
train_data = pd.read_csv(f"{path}/samsum-train.csv")
val_data = pd.read_csv(f"{path}/samsum-validation.csv")

In [5]:
train_data

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [6]:
train_data.shape

(14732, 3)

In [7]:
val_data.shape

(818, 3)

In [8]:
train_data.isnull().sum()

,0
id,0
dialogue,1
summary,0


In [9]:
val_data.isnull().sum()

,0
id,0
dialogue,0
summary,0


### Random Sampling
- we get some random data approx = 4000
- we will not going to take all data because is will take lot of time and computation power

In [10]:
# Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [11]:
train_data.shape

(4000, 3)

In [12]:
val_data.shape

(500, 3)

In [13]:
train_data.isnull().sum()

,0
id,0
dialogue,0
summary,0


## Data Pre-Processing

In [14]:
import re

def clean_data(text):
  # remove lines
  text = re.sub(r"\r\n", " ", text)
  # remove extra spaces
  text = re.sub(r"\s+", " ", text)
  # remove html tags
  text = re.sub(r"<.*?>", " ", text)
  # remove starting ending extra spaces & convert to lower-case
  text = text.strip().lower()

  return text

In [15]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)

In [16]:
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [17]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## Tokenize

In [18]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True) ## 150 is set because summary is smaller than dialogues

  inputs["labels"] = targets["input_ids"] ## token ids => add to input as labels

  return inputs

### we are converting it into list because it is compatible for huggingface transformer

In [20]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [21]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### input_ids - dialogue => token ids
- 1 means = EOS (End of Sequence)
- 0 means = padding

### attention mask
- 1 means = valid token
- 0 means = invalid token means we don't have to consider it at training time

### labels => target => summary token
- 1 means = EOS (End of Sequence)
- 0 means = padding

In [22]:
len(train_dataset[0]["input_ids"])

512

In [23]:
len(train_dataset[0]["labels"])

150

In [24]:
type(train_dataset)
type(val_dataset)

list

## Working with our Model

In [25]:
# NLP => generation task which is conditional generation means it is generated based on the input

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Fine Tune
- ye model already train hai bahut sare datas par, but
- apni problem ke hisab se isshe hum train karein ge taki ye humare input ka sahi output de


In [26]:
import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("device", device)
model.to(device)

device cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [27]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results", ## to save the model so that we don't have to train again-again

    num_train_epochs = 6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy = "epoch", ## means it is asking when we what to evaluate => ans after each epoch
    save_strategy = "epoch", ## when we want to save our model => ans after every epoch

    warmup_steps=500 ## means our initial learning rate is 0 and after warmup_steps learning rate will reach to its maximum value
)

In [28]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

### Training the Model
- Training process started

In [29]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.577381,0.380097
2,0.397485,0.359731
3,0.374499,0.354865
4,0.362143,0.350373
5,0.355740,0.349276
6,0.350639,0.349105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9029809265136719, metrics={'train_runtime': 1249.484, 'train_samples_per_second': 19.208, 'train_steps_per_second': 2.401, 'total_flos': 3248203235328000.0, 'train_loss': 0.9029809265136719, 'epoch': 6.0})

### The process involve to use any model
- model load => fine-tune => save the model

### Saving the Model
- there are two way to save the model
1. save_pretrained
2. from_pretrained

In [30]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

### to use the model when ever required

In [31]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Test the core logic for summarization

In [51]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) ## clean

  # tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"  ## this will return pytorch tensors
  ).to(device)

  # generate the text(summary) => which is in the form of token ids
  model.to(device)
  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,   ## in Ai we have an algorithm called beam search, num_beams=4 means our transformer will going to generate 4 sequences of output and finally give that summary which is best out of them
      early_stopping=True   ## jaise hi hume end of sequence mil jai wahi hume stop karjana hai
  )

  # token ids convert to text(summary) => decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True) ## skip_special_token means skip the EOS(end of sequ.), seperators, tab spaces, tags etc
  return summary

### Now we were going to test on some sample

In [43]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

In [52]:
summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.


In [37]:
!zip -r results.zip results

from google.colab import files
files.download("results.zip")


  adding: results/ (stored 0%)
  adding: results/checkpoint-1500/ (stored 0%)
  adding: results/checkpoint-1500/trainer_state.json (deflated 67%)
  adding: results/checkpoint-1500/config.json (deflated 63%)
  adding: results/checkpoint-1500/scheduler.pt (deflated 61%)
  adding: results/checkpoint-1500/training_args.bin (deflated 53%)
  adding: results/checkpoint-1500/rng_state.pth (deflated 26%)
  adding: results/checkpoint-1500/optimizer.pt (deflated 7%)
  adding: results/checkpoint-1500/generation_config.json (deflated 29%)
  adding: results/checkpoint-1500/model.safetensors (deflated 10%)
  adding: results/checkpoint-500/ (stored 0%)
  adding: results/checkpoint-500/trainer_state.json (deflated 58%)
  adding: results/checkpoint-500/config.json (deflated 63%)
  adding: results/checkpoint-500/scheduler.pt (deflated 62%)
  adding: results/checkpoint-500/training_args.bin (deflated 53%)
  adding: results/checkpoint-500/rng_state.pth (deflated 26%)
  adding: results/checkpoint-500/optimi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>